# Demo 02 : Customer support handoffs

We route a Veenhaven Bus service request to a specialist and preserve the request in shared graph state. Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · customer-support

A triage model chooses a specialist. The graph passes the same request to that specialist, which checks local records before answering.

In [2]:
import sqlite3
from pathlib import Path
from typing import Literal

from langchain.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from pydantic import BaseModel
from typing_extensions import TypedDict

DB = data("demo-world/service.db")
REQUEST = "Bus CE12-2021-0117 reports error F-CE-410. What should the mechanic do first, and is the part in stock?"

class Triage(BaseModel):
    specialist: Literal["manuals", "parts"]
    reason: str

class SupportState(TypedDict):
    request: str
    specialist: str
    reason: str
    evidence: list[str]
    answer: str

triage = llm.with_structured_output(Triage).invoke("Route the request to manuals for fault procedure or parts for stock. " + REQUEST)
print("handoff:", triage.specialist, "because", triage.reason)

handoff: manuals because The request asks what procedure to follow first for fault F-CE-410. Route to manuals for troubleshooting; the stock question can be handled by parts separately.


The first specialist may discover that another specialty is needed. A `Command` changes the destination while keeping the user request and collected evidence in state. Each specialist returns only its new records, and the handoff appends them once to what earlier specialists found.

In [3]:
def manuals(state):
    rows = []
    for path in sorted(Path(data_folder("demo-world/manuals")).glob("*.md")):
        text = path.read_text(encoding="utf-8")
        if "F-CE-410" in text or "CE-IM-04" in text:
            rows.append(f"[{path.name}] {text[:1200]}")
    return {"evidence": rows}

def parts(state):
    with sqlite3.connect(f"file:{Path(DB).as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no = 'CE-IM-04'").fetchall()
    fact = f"parts table: {rows}"
    return {"evidence": [fact]}

def route(state):
    return state["specialist"]

def specialist_node(name, function, next_agent):
    def run(state):
        update = function(state)
        evidence = state["evidence"] + update["evidence"]
        if next_agent:
            return Command(goto=next_agent, update={"evidence": evidence})
        return {"evidence": evidence}
    return run

def compose(state):
    answer = llm.invoke("Answer briefly using only the records below. Name the source file or table. "
                        f"Request: {state['request']}\nEvidence:\n" + "\n".join(state["evidence"])).content
    return {"answer": answer}

graph = StateGraph(SupportState)
graph.add_node("manuals", specialist_node("manuals", manuals, "parts"))
graph.add_node("parts", specialist_node("parts", parts, None))
graph.add_node("compose", compose)
graph.add_conditional_edges(START, route, ["manuals", "parts"])
graph.add_edge("parts", "compose")
graph.add_edge("compose", END)
initial = {"request": REQUEST, "specialist": triage.specialist, "reason": triage.reason,
           "evidence": [], "answer": ""}
result = graph.compile().invoke(initial)
print("evidence items:", len(result["evidence"]), "unique:", len(set(result["evidence"])))
print("evidence sources:", [item[1:].split("]")[0] if item.startswith("[") else item.split(":")[0] for item in result["evidence"]])
print("answer:\n", result["answer"])

evidence items: 3 unique: 3
evidence sources: ['ce-12-error-codes.md', 'ce-12-procedure-ce-im-04.md', 'parts table']
answer:
 Run inverter calibration first; replace the inverter module only if the dropout persists. The part is in stock: **CE-IM-04, 2 units in Broekdam**.

Sources: `ce-12-error-codes.md`, `ce-12-procedure-ce-im-04.md`, and the parts table.


The handoff preserves evidence gathered by the first specialist. The graph controls the destination and the final answer; a real support system would add a person before refunds, orders or account changes.